# Analyze and plot results for networks

This script takes in the results from run_over_regs.sh, assembles them into a single dataset, and performs analyses on the data.

In [1]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import matplotlib as mpl
import matplotlib.ticker as ticker

import numpy as np
import pandas as pd
from pydmd import DMD
import os
import plotly.graph_objects as go
import plotly.express as px

# import clustering packages
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
import seaborn as sns
from celluloid import Camera

plt.style.use('custom.mplstyle')
%config InlineBackend.figure_format = 'retina'
from tqdm import tqdm

from stoch_sim_model import *

## 0. Load data and build datasets

In [15]:
# Load data from infections
infection_type = 'prim'
reg_model = ''
runs = '-5-'
comment = "prim-Nact-Ediv-vir" #"full-reg-vir" #"act-reg-exp-reg" # mem-reg

d_mean = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/raw/stacked_data'+runs+'runs'+'-'+comment+'.pkl'
mean_df = pd.read_pickle(d_mean)

with pd.option_context('display.max_columns', None):
    display(mean_df)

,psi_Nact_I,psi_Nact_H,psi_Nact_IH,psi_NM_I,psi_NM_H,psi_NM_IH,psi_EM_I,psi_EM_H,psi_EM_IH,psi_Ediv_I,psi_Ediv_H,psi_Ediv_IH,d_I,K_IE,b_I,S_0,I_0,d_S,d_IE,d_IH,K_IH,A_init,b_Ain,b_H,d_H,K_EI,K_EH,N_0,max_Na,b_myc,d_myc,myc_thresh,t_act,t_bind,t_Na_div,t_E_div,t_M_div,t_EM_diff,t_E_die,t_E_cyt,p_load,s_load,T_max_pI,T_min_pI,harm_pI,harm_sI,harm_pS,harm_sS,max_pE,max_sE,T_max_pE,T_max_sE,inf_pM,inf_sM,init_M,int_pE,int_sE,int_pH,int_sH,min_pS,min_sS
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,-2.0,0.0,0.1,10000.0,7.500000e-08,10000000.0,1000.0,0.01,12.0,0.0,5000.0,1000.0,1.0,1.0,2.0,10000.0,5000.0,100.0,8.0,1592.428682,2.376505,398.107171,1.0,0.75,0.25,0.333333,0.5,15.0,2.5,0.666667,3.388366e+07,3.388366e+07,12.8448,1.3096,6.250986e+06,6.250986e+06,1.325664e+05,1.325664e+05,10584.8,10584.8,3.5200,3.5200,296.6,296.6,20.4,2.240333e+04,2.240333e+04,1.623202e+06,1.623202e+06,4.124600e+06,4.124600e+06
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,-2.0,0.0,0.1,10000.0,8.750000e-08,10000000.0,1000.0,0.01,12.0,0.0,5000.0,1000.0,1.0,1.0,2.0,10000.0,5000.0,100.0,8.0,1592.428682,2.376505,398.107171,1.0,0.75,0.25,0.333333,0.5,15.0,2.5,0.666667,3.784714e+07,3.784714e+07,11.8888,1.3656,6.410077e+06,6.410077e+06,1.832760e+05,1.832760e+05,15493.4,15493.4,3.7480,3.7480,452.4,452.4,14.2,3.117310e+04,3.117310e+04,1.830277e+06,1.830277e+06,3.991947e+06,3.991947e+06
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,-2.0,0.0,0.1,10000.0,1.000000e-07,10000000.0,1000.0,0.01,12.0,0.0,5000.0,1000.0,1.0,1.0,2.0,10000.0,5000.0,100.0,8.0,1592.428682,2.376505,398.107171,1.0,0.75,0.25,0.333333,0.5,15.0,2.5,0.666667,6.123564e+07,6.123564e+07,12.2992,0.8696,8.827547e+06,8.827547e+06,2.083153e+04,2.083153e+04,1791.0,1791.0,2.9040,2.9040,40.0,40.0,7.4,3.495526e+03,3.495526e+03,2.998429e+06,2.998429e+06,2.153164e+06,2.153164e+06
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,-2.0,0.0,0.1,10000.0,1.125000e-07,10000000.0,1000.0,0.01,12.0,0.0,5000.0,1000.0,1.0,1.0,2.0,10000.0,5000.0,100.0,8.0,1592.428682,2.376505,398.107171,1.0,0.75,0.25,0.333333,0.5,15.0,2.5,0.666667,8.350508e+07,8.350508e+07,12.3472,0.0000,1.123423e+07,1.123423e+07,1.234431e+04,1.234431e+04,1042.0,1042.0,2.9376,2.9376,26.6,26.6,6.0,2.262468e+03,2.262468e+03,4.110286e+06,4.110286e+06,1.729923e+05,1.729923e+05
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,-2.0,0.0,0.1,10000.0,1.250000e-07,10000000.0,1000.0,0.01,12.0,0.0,5000.0,1000.0,1.0,1.0,2.0,10000.0,5000.0,100.0,8.0,1592.428682,2.376505,398.107171,1.0,0.75,0.25,0.333333,0.5,15.0,2.5,0.666667,8.596707e+07,8.596707e+07,11.3232,0.0000,1.132416e+07,1.132416e+07,1.662225e+04,1.662225e+04,1332.2,1332.2,2.8176,2.8176,28.8,28.8,2.4,3.018197e+03,3.018197e+03,4.240656e+06,4.240656e+06,1.478555e+05,1.478555e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105995,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,0.0,0.5,500000.0,7.500000e-08,10000000.0,1000.0,0.01,12.0,0.0,5000.0,1000.0,1.0,1.0,2.0,500000.0,5000.0,100.0,8.0,1592.428682,2.376505,398.107171,1.0,0.75,0.25,0.333333,0.5,15.0,2.5,0.666667,1.700329e+06,1.700329e+06,25.0000,0.0000,1.200442e+06,1.200442e+06,0.000000e+00,0.000000e+00,0.0,0.0,0.0000,0.0000,0.0,0.0,0.0,0.000000e+00,0.000000e+00,3.846390e+05,3.846390e+05,8.852091e+06,8.852091e+06
105996,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,0.0,0.5,500000.0,8.750000e-08,10000000.0,1000.0,0.01,12.0,0.0,5000.0,1000.0,1.0,1.0,2.0,500000.0,5000.0,100.0,8.0,1592.428682,2.376505,398.107171,1.0,0.75,0.25,0.333333,0.5,15.0,2.5,0.666667,4.876463e+06,4.876463e+06,20.1784,14.2296,3.449914e+06,3.449914e+06,2.327731e+06,2.327731e+06,262494.2,262494.2,23.4096,23.4096,7928.8,7928.8,285.4,5.495536e+05,5.495536e+05,1.715497e+06,1.715497e+06,4.507528e+06,4.507528e+06
105997,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,0.0,0.5,500000.0,1.000000e-07,10000000.0,1000.0,0.01,12.0,0.0,5000.0,1000

## 1. Understanding the statistics of responses to an infection

In [16]:
# Create additional variables
K_IEs = np.unique(mean_df['K_IE'])
d_Is = np.unique(mean_df['d_I'])
b_S = np.mean(mean_df['d_S']*mean_df['S_0'])
N_0 = np.mean(mean_df['N_0'])
virs = np.unique(mean_df[['d_I','K_IE','b_I']].to_numpy(), axis = 0)
NM_reg = ['psi_NM_I', 'psi_NM_H', 'psi_NM_IH']
EM_reg = ['psi_EM_I', 'psi_EM_H', 'psi_EM_IH']
Nact_reg = ['psi_Nact_I', 'psi_Nact_H', 'psi_Nact_IH']
Ediv_reg = ['psi_Ediv_I', 'psi_Ediv_H', 'psi_Ediv_IH']

if "full-reg" in comment:
    reg = Nact_reg + NM_reg + EM_reg + Ediv_reg
    reg_label = param_names[-12:]
else:
    reg = (Nact_reg if "Nact" in comment else []) + (NM_reg if "NM" in comment else []) + (EM_reg if "EM" in comment else []) + (Ediv_reg if "Ediv" in comment else [])
    reg_label = (param_names[-12:-9] if "Nact" in comment else []) + (param_names[-9:-6] if "NM" in comment else []) + (param_names[-6:-3] if "EM" in comment else []) + (param_names[-3:] if "Ediv" in comment else [])
                                       

mean_df['int_presp'] = mean_df['int_pE'] + mean_df['inf_pM']
mean_df['int_sresp'] = mean_df['int_sE'] + mean_df['inf_sM']

# identify Biologically evidenced networks
mean_df['bio_reg'] = (mean_df[EM_reg[0]] > 0.0)*(mean_df[EM_reg[1]] > 0.0)*(mean_df[EM_reg[2]] > 0.0)*1

In [17]:
# save data sets
cutoff = 1 - 0.025 if 0.025*mean_df.shape[0]/len(virs) >= 1 else 1 - 10/(mean_df.shape[0]/len(virs))
infection_scenarios = []
no_eff_data = [[] for i in np.arange(len(virs))]

for l, (d_I, K_IE, b_I) in enumerate(virs):
    data = mean_df.loc[(mean_df["d_I"] == d_I)*(mean_df["K_IE"] == K_IE)*(mean_df["b_I"] == b_I), ['b_I','d_I', 'K_IE', 'I_0','S_0', 'N_0', 'd_S', 'bio_reg'] + Nact_reg + NM_reg + EM_reg + Ediv_reg + stat_names_for_df]

    # compute infection harm without T cell response
    no_eff_data[l] = lin_stoch_sim(N_0 = 0, I_0 = np.unique(data['I_0'])[0], K_IE = K_IE, d_I = d_I, b_I = b_I)
    no_eff_stats = no_eff_data[l]["summary_stats"]
    
    data.loc[:,"peff_protection"] = (no_eff_stats[4] - data['harm_pI'].to_numpy())/(b_S*(data['T_min_pI'] + sim_duration*(data['T_min_pI'] == 0)))
    data.loc[:,"seff_protection"] = (no_eff_stats[5] - data['harm_sI'].to_numpy())/(b_S*(data['T_min_pI'] + sim_duration*(data['T_min_pI'] == 0)))

    data.loc[:,"peff_toxicity"] = data['harm_pS'].to_numpy()/(b_S*(data['T_min_pI'] + sim_duration*(data['T_min_pI'] == 0)))
    data.loc[:,"seff_toxicity"] = data['harm_sS'].to_numpy()/(b_S*(data['T_min_pI'] + sim_duration*(data['T_min_pI'] == 0)))

    data.loc[:,"peff_utility"] = (no_eff_stats[4] + 1.0*no_eff_stats[6] - (data['harm_pI'] + data['harm_pS']).to_numpy())/(b_S*(data['T_min_pI'] + sim_duration*(data['T_min_pI'] == 0)))
    data.loc[:,"seff_utility"] = (no_eff_stats[5] + 1.0*no_eff_stats[7] - (data['harm_sI'] + data['harm_sS']).to_numpy())/(b_S*(data['T_min_pI'] + sim_duration*(data['T_min_pI'] == 0)))

    data.loc[:,"peff_efficiency"] = (data['peff_protection'] > 0)*(data['peff_protection'] - data['peff_toxicity'])/data['peff_protection']
    data.loc[:,"seff_efficiency"] = (data['seff_protection'] > 0)*(data['seff_protection'] - data['seff_toxicity'])/data['seff_protection']

    if 'comp_model' not in comment:
        data.loc[:, "high_putility"] = 1*(data['peff_utility'] >= np.quantile(data['peff_utility'], cutoff))
        data.loc[:, "high_ptoxicity"] = 1*(-data['peff_toxicity'] >= np.quantile(-data['peff_toxicity'], cutoff))
        data.loc[:, "high_presponse"] = 1*(data['int_pE'] >= np.quantile(data['int_pE'], cutoff))
        data.loc[:, "high_pefficiency"] = 1*(data['peff_efficiency'] >= np.quantile(data['peff_efficiency'], cutoff))
        data.loc[:, "high_pmemory"] = 1*(data['init_M'] >= np.quantile(data['init_M'], cutoff))
        data.loc[:, "high_clear_timing"] = 1*(-data['T_max_pI'] >= np.quantile(-data['T_max_pI'], cutoff))
        data.loc[:, "high_resp_timing"] = 1*(-data['T_max_pE'] >= np.quantile(-data['T_max_pE'], cutoff))
        data.loc[:, "high_sutility"] = 1*(data['seff_utility'] >= np.quantile(data['seff_utility'], cutoff))
        data.loc[:, "high_stoxicity"] = 1*(-data['seff_toxicity'] >= np.quantile(-data['seff_toxicity'], cutoff))
        data.loc[:, "high_sresponse"] = 1*(data['int_sE'] >= np.quantile(data['int_sE'], cutoff))
        data.loc[:, "high_sefficiency"] = 1*(data['seff_efficiency'] >= np.quantile(data['seff_efficiency'], cutoff))
        
    infection_scenarios.append(data)

# stack datasets
clustered_mean_df = pd.concat(infection_scenarios)

clustered_mean_df.to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/processed_data'+runs+'runs'+'-'+comment+'.pkl')

/opt/minimamba/envs/maximmune/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/minimamba/envs/maximmune/lib/python3.11/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/minimamba/envs/maximmune/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/minimamba/envs/maximmune/lib/python3.11/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/minimamba/envs/maximmune/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/minimamba/envs/maximmune/lib/python3.11/site-packages/numpy/core/_methods.py:192: RuntimeWar